In [ ]:
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import sys
sys.path.append('../')

In [ ]:
from constants import ISCED_mapping_short, LMS_mapping, SIE_mapping, brussels_codes_muni_dict
from constants import PT_COLOR, BIKE_COLOR

# Rename colors to make it clearer
PURPLE = PT_COLOR
ORANGE = BIKE_COLOR

# 1 Brussels population generation

In [ ]:
# Load data
synthetic_pop = pd.read_csv("output/synthetic_population_output.csv")
age5_sector = pd.read_csv("input_data/TF_CENSUS_2021_S01_cleaned.csv")
edu_sector = pd.read_csv("input_data/TF_CENSUS_2021_S11_cleaned.csv")
emp_sector = pd.read_csv("input_data/TF_CENSUS_2021_S12_cleaned.csv")
prov_age5_edu = pd.read_csv("input_data/TF_CENSUS_2021_HC04_3_cleaned.csv")
prov_age5_emp = pd.read_csv("input_data/TF_CENSUS_2021_HC04_1_cleaned.csv")
# prov_age5_edu_emp = pd.read_csv("input_data/TF_CENSUS_2021_HC23_2_cleaned.csv")

In [ ]:
# Validation of synthetic population against census inputs
def build_comparison(synth_df, census_df, keys, synth_col="population", census_col="MS_POP"):
    comp = (
        synth_df.groupby(keys, as_index=False)[synth_col].sum()
        .merge(
            census_df.groupby(keys, as_index=False)[census_col].sum(),
            on=keys,
            how="outer",
        )
        .fillna(0)
    )
    comp = comp.rename(columns={synth_col: "synthetic", census_col: "census"})
    comp["diff"] = comp["synthetic"] - comp["census"]
    comp["abs_diff"] = comp["diff"].abs()
    comp["pct_diff"] = np.where(comp["census"] > 0, 100 * comp["diff"] / comp["census"], np.nan)
    return comp


# 1) Age distribution per gender
synth_age_gender = synthetic_pop.rename(columns={"sex": "CD_SEX", "age": "CD_AGE"})
age_gender_comp = build_comparison(
    synth_df=synth_age_gender,
    census_df=age5_sector,
    keys=["CD_SEX", "CD_AGE"],
    synth_col="population",
    census_col="MS_POP",
)

# 2) Education distribution per gender
synth_edu_gender = synthetic_pop.rename(columns={"sex": "CD_SEX", "education": "CD_EDU"})
edu_gender_comp = build_comparison(
    synth_df=synth_edu_gender,
    census_df=edu_sector,
    keys=["CD_SEX", "CD_EDU"],
    synth_col="population",
    census_col="MS_POP",
)

# 3) Employment distribution per gender
synth_emp_gender = synthetic_pop.rename(columns={"sex": "CD_SEX", "employment": "CD_EMPLOYMENT"})
emp_gender_comp = build_comparison(
    synth_df=synth_emp_gender,
    census_df=emp_sector,
    keys=["CD_SEX", "CD_EMPLOYMENT"],
    synth_col="population",
    census_col="MS_POP",
)

# 4) Total population per statistical sector
synth_sector_total = synthetic_pop.rename(columns={"sector": "CD_SECTOR"})
sector_total_comp = build_comparison(
    synth_df=synth_sector_total,
    census_df=age5_sector,
    keys=["CD_SECTOR"],
    synth_col="population",
    census_col="MS_POP",
)

# 5) Total population per municipality
synth_muni_total = synthetic_pop.assign(
    CD_MUNI=synthetic_pop["sector"].astype(str).str[:5]
)
census_muni_total = age5_sector.assign(
    CD_MUNI=age5_sector["CD_SECTOR"].astype(str).str[:5]
)

muni_total_comp = build_comparison(
    synth_df=synth_muni_total,
    census_df=census_muni_total,
    keys=["CD_MUNI"],
    synth_col="population",
    census_col="MS_POP",
)

# 6) Municipality distribution per gender
synth_muni_gender = synth_muni_total.rename(columns={"sex": "CD_SEX"})
muni_gender_comp = build_comparison(
    synth_df=synth_muni_gender,
    census_df=census_muni_total,
    keys=["CD_SEX", "CD_MUNI"],
    synth_col="population",
    census_col="MS_POP",
)

def print_summary(name, comp_df):
    total_census = comp_df["census"].sum()
    total_synth = comp_df["synthetic"].sum()
    total_abs_diff = comp_df["abs_diff"].sum()
    wmape = (total_abs_diff / total_census * 100) if total_census > 0 else np.nan

    print(f"\n=== {name} ===")
    print(f"Total census   : {total_census:,.0f}")
    print(f"Total synthetic: {total_synth:,.0f}")
    print(f"Total abs diff : {total_abs_diff:,.0f}")
    print(f"wMAPE (%)      : {wmape:.4f}")
    print("Top 10 percentage differences:")
    print(comp_df.sort_values("pct_diff", ascending=False).head(10))


print_summary("Age distribution per gender", age_gender_comp)
print_summary("Education distribution per gender", edu_gender_comp)
print_summary("Employment distribution per gender", emp_gender_comp)
print_summary("Total population per sector", sector_total_comp)
print_summary("Total population per municipality", muni_total_comp)
print_summary("Municipality distribution per gender", muni_gender_comp)

# Optional: keep all outputs in one container for later use
validation_results = {
    "age_gender": age_gender_comp,
    "edu_gender": edu_gender_comp,
    "emp_gender": emp_gender_comp,
    "sector_total": sector_total_comp,
    "muni_total": muni_total_comp,
    "muni_gender": muni_gender_comp,
}

In [ ]:
sexes = sorted(prov_age5_edu["CD_SEX"].unique())
educations = sorted(prov_age5_edu["CD_EDU"].unique())
employments = sorted(prov_age5_emp["CD_EMPLOYMENT"].unique())

In [ ]:
def format_muni_label(code):
    # robust conversion for values like 21001, "21001", or 21001.0
    try:
        code_int = int(float(code))
    except (ValueError, TypeError):
        return str(code)
    name = brussels_codes_muni_dict.get(code_int, "Unknown")
    # return f"{name}\n({code_int})"
    return f"{name}"

In [ ]:
def format_compact(x, pos=None):
    x = float(x)
    ax = abs(x)

    if ax >= 1_000_000:
        v = x / 1_000_000
        return f"{v:.1f}".rstrip("0").rstrip(".") + "M"
    if ax >= 1_000:
        v = x / 1_000
        return f"{v:.1f}".rstrip("0").rstrip(".") + "k"
    return f"{int(x):,}"

def plot_validation_grid(plot_specs, main_title, legend_ncol=6, figure_name=None):
    fig, axes = plt.subplots(len(plot_specs), 1, figsize=(16, 6*len(plot_specs)), sharey=False)
    line_colors = {"M": ORANGE, "F": PURPLE}
    markers = {"M": "o", "F": "^"}
    # bar_styles = {"M": "/", "F": "."}

    legend_handles = []
    legend_labels = []

    for ax, spec in zip(axes, plot_specs):
        comp_df = spec["comp_df"]
        category_col = spec["category_col"]
        x_label = spec["x_label"]
        category_order = spec.get("category_order", None)
        tick_label_fn = spec.get("tick_label_fn", None)

        sexes_order = [s for s in sexes if s in comp_df["CD_SEX"].unique()]
        source_order = ["synthetic", "census"]

        cats = category_order if category_order is not None else sorted(comp_df[category_col].unique())
        x = np.arange(len(cats))
        groups = [(sx, src) for sx in sexes_order for src in source_order]
        width = 0.8 / max(len(groups), 1)

        for i, (sx, src) in enumerate(groups):
            sub = comp_df[comp_df["CD_SEX"] == sx][[category_col, src]].drop_duplicates()
            vals = sub.set_index(category_col)[src].reindex(cats).fillna(0).values
            offset = (i - (len(groups) - 1) / 2) * width
            ax.bar(
                x + offset,
                vals,
                width=width,
                label=f"{sx} - {src}",
                color=ORANGE if sx == "M" else PURPLE,
                alpha=1 if src == "synthetic" else 0.5,
                # hatch=bar_styles.get(sx, "/")
            )

        ax.set_xlabel(x_label)
        ax.set_ylabel("Population")
        ax.set_xticks(x)
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(format_compact))

        tick_labels = cats if tick_label_fn is None else [tick_label_fn(c) for c in cats]
        ax.set_xticklabels(tick_labels, rotation=45, ha="right")
        ax.grid(axis="y", alpha=0.3)

        ax2 = ax.twinx()
        for sx in sexes_order:
            sub = (
                comp_df[comp_df["CD_SEX"] == sx][[category_col, "synthetic", "census"]]
                .drop_duplicates()
                .set_index(category_col)
                .reindex(cats)
                .fillna(0)
            )
            syn = sub["synthetic"].to_numpy(dtype=float)
            cen = sub["census"].to_numpy(dtype=float)
            rel_err = np.where(cen > 0, 100.0 * (syn - cen) / cen, np.nan)

            ax2.plot(
                x,
                rel_err,
                linestyle="--",
                marker=markers.get(sx, "o"),
                linewidth=1.5,
                markersize=5,
                color=line_colors.get(sx, "black"),
                label=f"{sx} rel. error (%)",
            )

        ax2.axhline(0, color="gray", linewidth=1, alpha=0.7)
        ax2.set_ylabel("Relative error (%)")

        h1, l1 = ax.get_legend_handles_labels()
        h2, l2 = ax2.get_legend_handles_labels()
        legend_handles.extend(h1 + h2)
        legend_labels.extend(l1 + l2)

    unique = {}
    for h, la in zip(legend_handles, legend_labels):
        unique[la] = h

    fig.suptitle(main_title, y=0.99, fontsize=18)
    fig.legend(
        unique.values(),
        unique.keys(),
        loc="upper center",
        ncol=legend_ncol,
        frameon=False,
        bbox_to_anchor=(0.5, 0.965),
    )

    plt.tight_layout(rect=(0, 0, 1, 0.965))
    plt.show()

    out_path = f"figures/{figure_name}_with_relative_errors.png" if figure_name else "figures/multiplot_with_relative_errors.png"

    fig.savefig(out_path, dpi=300, bbox_inches="tight")
    print(f"Saved figure to: {out_path}")

with plt.rc_context({
    "font.size": 16,
    "axes.titlesize": 18,
    "axes.labelsize": 16,
    "xtick.labelsize": 13,
    "ytick.labelsize": 13,
    "legend.fontsize": 14,
    "figure.titlesize": 22,
}):
    plot_validation_grid(
        [
            {
                "comp_df": age_gender_comp,
                "category_col": "CD_AGE",
                "x_label": "Age bracket",
                "category_order": [
                    "0-4", "5-9", "10-14", "15-19", "20-24",
                    "25-29", "30-34", "35-39", "40-44", "45-49",
                    "50-54", "55-59", "60-64", "65-69", "70-74",
                    "75-79", "80-84", "85-89", "90-94", "95-99", "100+",
                ],
            },
            {
                "comp_df": edu_gender_comp,
                "category_col": "CD_EDU",
                "x_label": "Educational attainment",
                "category_order": educations,
                "tick_label_fn": lambda code: ISCED_mapping_short.get(code, f"Unknown ({code})"),
            },
            {
                "comp_df": emp_gender_comp,
                "category_col": "CD_EMPLOYMENT",
                "x_label": "Labour market status",
                "category_order": employments,
                "tick_label_fn": lambda code: LMS_mapping.get(code, f"Unknown ({code})"),
            },
            {
                "comp_df": muni_gender_comp,
                "category_col": "CD_MUNI",
                "x_label": "Municipality",
                "category_order": sorted(muni_total_comp["CD_MUNI"].unique()),
                "tick_label_fn": format_muni_label,
            },
        ],
        main_title="Synthetic population vs Census 2021 by gender",
        figure_name="multiplot_all",
        legend_ncol=6,
    )

### Employment status stats

In [ ]:
synthetic_pop.groupby('employment', as_index=False)['population'].sum()

# 2 Active population attributes

In [ ]:
emp_enriched = pd.read_csv("output/employed_population.csv")

hc05_6 = pd.read_csv("input_data/TF_CENSUS_2021_HC05_6_cleaned.csv")
hc05_7 = pd.read_csv("input_data/TF_CENSUS_2021_HC05_7_cleaned.csv")

In [ ]:
# 1) Professional status distribution per gender
synth_prof_gender = emp_enriched.rename(columns={"sex": "CD_SEX", "professional_status": "CD_PROF_STATUS"})
prof_gender_comp = build_comparison(
    synth_df=synth_prof_gender,
    census_df=hc05_6,
    keys=["CD_SEX", "CD_PROF_STATUS"],
    synth_col="population",
    census_col="MS_POP",
)

# 2) Industry distribution per gender
synth_ind_gender = emp_enriched.rename(columns={"sex": "CD_SEX", "industry": "CD_INDUSTRY"})
ind_gender_comp = build_comparison(
    synth_df=synth_ind_gender,
    census_df=hc05_6,
    keys=["CD_SEX", "CD_INDUSTRY"],
    synth_col="population",
    census_col="MS_POP",
)

print_summary("Professional status distribution per gender", prof_gender_comp)
print_summary("Industry distribution per gender", ind_gender_comp)

In [ ]:
def format_industry_label(code):
    if code == 'M_N':
        return "M-N"
    return code

with plt.rc_context({
    "font.size": 16,
    "axes.titlesize": 18,
    "axes.labelsize": 16,
    "xtick.labelsize": 13,
    "ytick.labelsize": 13,
    "legend.fontsize": 14,
    "figure.titlesize": 22,
}):
    plot_validation_grid(
        [
            {
                "comp_df": prof_gender_comp,
                "category_col": "CD_PROF_STATUS",
                "x_label": "Status in employment",
                "tick_label_fn": lambda code: SIE_mapping.get(code, f"Unknown ({code})"),
            },
            {
                "comp_df": ind_gender_comp,
                "category_col": "CD_INDUSTRY",
                "x_label": "Economic sector",
                "tick_label_fn": format_industry_label,
            },
        ],
        main_title="Employed synthetic population vs Census 2021 by gender",
        figure_name="multiplot_emp",
        legend_ncol=6,
    )

In [ ]:
# Validation: joint distribution of industry x professional status

synth_prof_ind = emp_enriched.rename(
    columns={
        "sex": "CD_SEX",
        "industry": "CD_INDUSTRY",
        "professional_status": "CD_PROF_STATUS",
    }
)

# 1) By sex + industry + professional status
prof_ind_sex_comp = build_comparison(
    synth_df=synth_prof_ind,
    census_df=hc05_6,
    keys=["CD_SEX", "CD_INDUSTRY", "CD_PROF_STATUS"],
    synth_col="population",
    census_col="MS_POP",
)

# 2) Overall industry + professional status (all sexes combined)
prof_ind_total_comp = build_comparison(
    synth_df=synth_prof_ind,
    census_df=hc05_6,
    keys=["CD_INDUSTRY", "CD_PROF_STATUS"],
    synth_col="population",
    census_col="MS_POP",
)

print_summary("Industry x Professional status (by sex)", prof_ind_sex_comp)
print_summary("Industry x Professional status (overall)", prof_ind_total_comp)

print("\nTop 20 absolute differences (by sex):")
print(
    prof_ind_sex_comp.sort_values("abs_diff", ascending=False)
    .head(20)[["CD_SEX", "CD_INDUSTRY", "CD_PROF_STATUS", "synthetic", "census", "diff", "pct_diff"]]
)

# Keep for later reuse
validation_results["prof_ind_sex"] = prof_ind_sex_comp
validation_results["prof_ind_total"] = prof_ind_total_comp

### Status in employment stats

In [ ]:
emp_enriched.groupby('professional_status', as_index=False)['population'].sum()

### Age stats

In [ ]:
# Get the median lower bracket of age of the synthetic population
emp_enriched['age_lower'] = emp_enriched['age'].str.split('-').str[0].str.replace('+', '').astype(int)

emp_enriched.groupby('age_lower', as_index=False)['population'].sum().median()

### Gender stats

In [ ]:
# Check the distribution of the sex variable in the synthetic population
# In total numbes and percentages
sex_grouping = emp_enriched.groupby('sex', as_index=False)['population'].sum()

print(sex_grouping)
print(sex_grouping['population']/emp_enriched['population'].sum()*100)

# 0 STATISTICAL SECTORS Stats

In [ ]:
# Number of statistical sectors in emp_enriched
n_sectors = emp_enriched["sector"].nunique()
print(f"Number of statistical sectors: {n_sectors}")

# Agents per sector stats
pop_per_sector = (
    emp_enriched.groupby("sector", as_index=False)["population"]
    .sum()
    .rename(columns={"population": "population_sum"})
    .sort_values("population_sum", ascending=False)
)
print("\nPopulation sum per sector:")
print(pop_per_sector.describe())

print("\nTop 10 sectors by population:")
print(pop_per_sector.head(10))

In [ ]:
# Number of statistical sectors in the whole popluation
whole_population = pd.read_csv("output/synthetic_population_output.csv")

n_sectors = whole_population["sector"].nunique()
print(f"Number of statistical sectors: {n_sectors}")

# Agents per sector stats
pop_per_sector = (
    whole_population.groupby("sector", as_index=False)["population"]
    .sum()
    .rename(columns={"population": "population_sum"})
    .sort_values("population_sum", ascending=False)
)
print("\nPopulation sum per sector:")
print(pop_per_sector.describe())

print("\nTop 10 sectors by population:")
print(pop_per_sector.head(10))